# Week 5 — Embeddings: A Map of Meaning  (fully worked)

**The heart of the course.** You'll turn each item — a passage or a picture — into a *vector*, a
few hundred numbers whose position is learned from the company it keeps. Similar things land
near each other, so your corpus **sorts itself by meaning** and a finding counting couldn't give
you appears. Then the catch: switch PCA to t-SNE and the same data rearranges — a chart is a
*choice*.

> Errors? Last line → paste back to the AI → `../kits/common-errors-cheatsheet.md`.

In [ ]:
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q sentence-transformers scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Load a sentence-embedding model

`sentence-transformers` downloads a small model (`all-MiniLM-L6-v2`) the first time. On Colab,
attach a GPU (`Runtime → Change runtime type → T4`) for speed; CPU works for small corpora.
If the model can't be downloaded (e.g. in the offline harness), we fall back to a simple
stand-in embedder so the *rest of the pipeline* still runs — clearly marked, not for real use.

In [ ]:
def get_text_embedder():
    """Return a function texts->vectors. Real model when possible; stand-in offline."""
    if not SMOKE:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("all-MiniLM-L6-v2")
            print("using real model: all-MiniLM-L6-v2")
            return lambda texts: np.asarray(model.encode(list(texts)))
        except Exception as e:
            print("could not load real model, using offline stand-in:", e)
    # Offline stand-in: hashed bag-of-words -> deterministic vector. NOT a real embedding;
    # it only lets the plotting/clustering code run in the compatibility harness.
    print("using OFFLINE STAND-IN embedder (harness only — not real embeddings)")
    rng_dim = 64
    def embed(texts):
        out = np.zeros((len(texts), rng_dim))
        for i, t in enumerate(texts):
            for tok in t.lower().split():
                out[i, hash(tok) % rng_dim] += 1.0
        norms = np.linalg.norm(out, axis=1, keepdims=True)
        return out / np.where(norms == 0, 1, norms)
    return embed

embed_text = get_text_embedder()

## Embed your own corpus

In class this is *your* data — a beloved artist's songs, fan posts, museum blurbs. Here we use a
tiny themed set so clusters are visible. Replace `corpus` with your own list of strings.

In [ ]:
corpus = [
    # "space" cluster
    "the rocket lifted off toward the distant red planet",
    "astronauts floated in orbit above the blue earth",
    "a telescope spotted a new moon around saturn",
    # "cooking" cluster
    "fold the butter into the flour until crumbly",
    "simmer the tomatoes with garlic and fresh basil",
    "whisk the eggs then pour the batter into the pan",
    # "heartbreak" cluster
    "you left and the silence in the house got loud",
    "i kept your number but i never call it now",
    "we were a song that the radio stopped playing",
]
vecs = embed_text(corpus)
print("embedded", len(corpus), "items into", vecs.shape[1], "dimensions")

## Plot it — PCA, then t-SNE (the chart is a choice)

In [ ]:
def plot_2d(coords, labels, title):
    plt.figure(figsize=(6,5))
    plt.scatter(coords[:,0], coords[:,1])
    for (x, y), lab in zip(coords, labels):
        plt.annotate(lab[:18], (x, y), fontsize=7)
    plt.title(title); plt.tight_layout(); plt.show()

pca = PCA(n_components=2).fit_transform(vecs)
plot_2d(pca, corpus, "PCA — a linear flattening")

In [ ]:
# t-SNE makes a DIFFERENT choice about how to flatten. Same data, rearranged.
n = len(corpus)
tsne = TSNE(n_components=2, perplexity=min(5, n-1), random_state=0, init="pca").fit_transform(vecs)
plot_2d(tsne, corpus, "t-SNE — clusters tighten, distances shift")

## Hunt the surprise cluster

"Near" means "probably similar"; exact distances mean little. Find each item's nearest
neighbours and look for a grouping you didn't expect — that's your candidate finding.

In [ ]:
def nearest(i, k=2):
    sims = vecs @ vecs[i]
    order = sims.argsort()[::-1]
    return [j for j in order if j != i][:k]

probe = 0
print("Item:", corpus[probe])
for j in nearest(probe):
    print("  near:", corpus[j])

## The image path — the same map, for pictures

Image projects embed *pictures* with a CLIP model (`clip-ViT-B-32`) and watch them group by
visual style nobody tagged. The code is the same shape as the text path; only the embedder
changes. (Offline, we fall back to a color-feature stand-in so the pipeline runs.)

In [ ]:
from PIL import Image
def swatch(rgb, s=32):
    a = np.zeros((s, s, 3), dtype=np.uint8); a[:, :] = rgb; return Image.fromarray(a)
images = [swatch(c) for c in [(10,10,40),(20,20,60),(240,180,40),(250,200,60),(30,90,50),(40,110,60)]]

def get_image_embedder():
    if not SMOKE:
        try:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer("clip-ViT-B-32")
            print("using real CLIP model")
            return lambda imgs: np.asarray(m.encode(imgs))
        except Exception as e:
            print("CLIP unavailable, using color stand-in:", e)
    print("using OFFLINE color stand-in (harness only)")
    return lambda imgs: np.asarray([np.asarray(im).reshape(-1,3).mean(0)/255 for im in imgs])

img_vecs = get_image_embedder()(images)
ip = PCA(n_components=2).fit_transform(img_vecs)
plt.figure(figsize=(5,4)); plt.scatter(ip[:,0], ip[:,1])
plt.title("Images cluster by visual style"); plt.tight_layout(); plt.show()

In [ ]:
# Save your embeddings to Drive so Week 6 can reuse them without recomputing.
import os, numpy as np
_out = os.path.join(PROJECT_DIR, "week05_embeddings.npy")
np.save(_out, vecs)
print("saved embeddings ->", _out)

## You made a map of meaning

Your corpus placed itself by meaning, and you saw the same data tell two stories under PCA vs.
t-SNE. That tension — *the picture made a choice* — is the reading skill you'll turn on your own
charts in Week 9.

**Sketch:** toggle PCA vs. t-SNE and screenshot how the picture changes. Name one neighbour (or
image cluster) that surprised you — real pattern, or projection artifact?